# Aula de Ciência de Dados para Devs: Limpeza e Preparação de Dados

**Bem-vindo(a) à Parte 2: A Prática!**

Se na teoria tudo parece fazer sentido, é na prática que o conhecimento realmente se fixa. Nesta aula, você vai atuar como um detetive de dados. Vamos pegar um arquivo de cadastro de usuários de um e-commerce fictício (`usuarios_sujo.csv`), que está cheio de problemas comuns do mundo real, e vamos aplicar técnicas sistemáticas para limpá-lo e organizá-lo.

Ao final, teremos um dataset íntegro, confiável e pronto para a próxima fase: a análise exploratória de dados, onde poderemos extrair insights valiosos.

**Ferramentas que usaremos:**
- **Python:** Nossa linguagem de programação principal.
- **Pandas:** A biblioteca essencial para manipulação e análise de dados em Python. Pense nela como uma planilha superpoderosa que você controla com código.
- **NumPy:** Usada pelo Pandas por baixo dos panos, é fundamental para operações numéricas eficientes.
- **Faker:** Para gerar dados fictícios e criar nosso próprio dataset "sujo".

### Nosso Fluxo de Trabalho

Para nos guiar, seguiremos um fluxo de trabalho estruturado. Pense nisso como um mapa que nos levará do caos à ordem:

**1. Fonte de Dados Brutos (`usuarios_sujo.csv`)**
   - Nosso ponto de partida. Um arquivo com dados realistas, porém problemáticos.
     
**2. Carregamento e Inspeção Inicial (Profiling)**
   - Carregar os dados em um DataFrame e usar ferramentas de diagnóstico para identificar os problemas.
     
**3. Ciclo de Limpeza (Transformação)**
   - Esta é a fase iterativa onde aplicamos as correções:
     - Tratar Valores Nulos
     - Corrigir Tipos de Dados
     - Remover Duplicatas
     - Padronizar Dados Categóricos
     
**4. Validação**
   - Verificar se a limpeza foi bem-sucedida e se os dados agora fazem sentido.
     
**5. Saída de Dados Limpos (`usuarios_limpo.csv`)**
   - Salvar nosso trabalho em um novo arquivo, pronto para ser usado em análises futuras.

---
## 1. Setup do Ambiente e Geração dos Dados

Primeiro, vamos importar as bibliotecas necessárias. Em seguida, vamos criar nosso próprio arquivo `usuarios_sujo.csv`. Isso garante que todos tenham exatamente o mesmo ponto de partida e que o notebook seja autocontido.

**Tipos de problemas que vamos introduzir:**
- **Valores Faltantes:** `email` e `valor_ultima_compra` terão valores nulos (`NaN`).
- **Tipos de Dados Incorretos:** `valor_ultima_compra` será uma string (object) com símbolos e vírgulas, e as colunas de data serão strings.
- **Formatos Inconsistentes:** A coluna `data_cadastro` terá múltiplos formatos de data (ex: 'YYYY-MM-DD', 'DD/MM/YYYY').
- **Dados Categóricos Não Padronizados:** A coluna `estado` terá variações como 'SP', 'São Paulo' e 'sao paulo'.
- **Linhas Duplicadas:** Inseriremos algumas linhas completamente duplicadas.

In [17]:
%pip install faker
# Importando as bibliotecas
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime

# Inicializando o Faker para gerar dados em português
fake = Faker('pt_BR')

# Função para gerar os dados sujos
def gerar_dados_sujos(num_usuarios=200):
    dados = []

    # Formatos de data que vamos misturar
    formatos_data = ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%d-%b-%Y']

    # Variações para os estados
    estados_sp = ['SP', 'São Paulo', 'sao paulo']
    estados_rj = ['RJ', 'Rio de Janeiro', 'rio de janeiro']
    outros_estados = ['MG', 'PR', 'BA', 'SC']

    for i in range(num_usuarios):
        # Introduzindo valores faltantes (NaN) de forma aleatória
        email = fake.email() if random.random() > 0.1 else np.nan # 10% de chance de email nulo
        valor_compra = round(random.uniform(10, 1000), 2) if random.random() > 0.15 else np.nan # 15% de chance de valor nulo

        # Formatando o valor da compra como string com inconsistências
        if pd.notna(valor_compra):
            valor_compra_str = f"R$ {valor_compra:.2f}".replace('.', ',')
        else:
            # Adicionando outras strings não numéricas para sujar mais
            valor_compra_str = np.nan if random.random() > 0.3 else 'Não informado'

        # Escolhendo um formato de data aleatório
        formato_escolhido = random.choice(formatos_data)
        data_cadastro = fake.date_between(start_date='-2y', end_date='today').strftime(formato_escolhido)

        # Escolhendo um estado com inconsistências
        if i % 4 == 0:
            estado = random.choice(estados_sp)
        elif i % 7 == 0:
            estado = random.choice(estados_rj)
        else:
            estado = random.choice(outros_estados)

        dado = {
            'user_id': fake.uuid4(),
            'nome': fake.name(),
            'email': email,
            'data_cadastro': data_cadastro,
            'cidade': fake.city(),
            'estado': estado,
            'valor_ultima_compra': valor_compra_str,
            'data_ultimo_login': fake.date_time_between(start_date='-30d', end_date='now')
        }
        dados.append(dado)

    df = pd.DataFrame(dados)

    # Introduzindo linhas duplicadas
    duplicatas = df.sample(n=15, random_state=42)
    df_final = pd.concat([df, duplicatas]).reset_index(drop=True)

    return df_final

# Gerar e salvar o arquivo CSV
df_sujo = gerar_dados_sujos()
df_sujo.to_csv('usuarios_sujo.csv', index=False)

print("Arquivo 'usuarios_sujo.csv' gerado com sucesso!")
print(f"Total de linhas: {len(df_sujo)}")

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Arquivo 'usuarios_sujo.csv' gerado com sucesso!
Total de linhas: 215



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 2. Passo a Passo do Exercício Prático

Agora começa a nossa missão! Vamos carregar o arquivo `usuarios_sujo.csv` que acabamos de criar e seguir nosso fluxo de trabalho para limpá-lo passo a passo.

### 2.1 Carregamento dos Dados

Usaremos a função `pd.read_csv()` do Pandas para ler nosso arquivo e carregá-lo em uma estrutura de dados chamada **DataFrame**. Pense no DataFrame como uma tabela ou planilha dentro do Python.

In [18]:
# Carregando o arquivo CSV para um DataFrame
df_usuarios = pd.read_csv('usuarios_sujo.csv')

# Exibindo as dimensões do DataFrame (linhas, colunas)
print(f"O dataset possui {df_usuarios.shape[0]} linhas e {df_usuarios.shape[1]} colunas.")

O dataset possui 215 linhas e 8 colunas.


### 2.2 Inspeção Inicial (Data Profiling)

Antes de sair corrigindo tudo, precisamos entender a "cena do crime". O Data Profiling é o processo de investigar o dataset para entender sua estrutura, qualidade e conteúdo. É aqui que identificamos os problemas.

#### Usando `.head()`, `.tail()` e `.sample()` para Amostragem

Essas funções nos permitem "espiar" os dados de diferentes ângulos:
- `.head(n)`: Mostra as primeiras `n` linhas (o padrão é 5).
- `.tail(n)`: Mostra as últimas `n` linhas (o padrão é 5).
- `.sample(n)`: Mostra uma amostra aleatória de `n` linhas, útil para ter uma visão imparcial do dataset.

In [19]:
print("--- Primeiras 5 linhas ---")
display(df_usuarios.head())

print("\n--- Últimas 5 linhas ---")
display(df_usuarios.tail())

print("\n--- Amostra aleatória de 5 linhas ---")
display(df_usuarios.sample(5))

--- Primeiras 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
0,ba09f178-c78a-412e-a548-d10f2ee8ab2a,Marcos Vinicius Silveira,monteiromateus@example.org,2025-10-09,Pereira,São Paulo,NaN,2026-08-01 01:52:23
1,e1eb3936-dc32-4972-af19-8e1b38e5cebf,Srta. Isabel Vieira,kaiqueguerra@example.com,01-25-2025,Correia da Mata,MG,"R$ 647,44",2026-07-21 22:17:20
2,7b8953c0-1c26-4da3-b067-e79da74c0260,Valentim da Conceição,limavitoria@example.org,2024-12-20,Viana do Oeste,PR,"R$ 618,83",2026-08-15 13:00:25
3,dfd46842-e520-47ae-8c56-2d3671affc96,Sr. Théo Cavalcanti,mateus40@example.net,2025-06-28,Cirino da Praia,BA,"R$ 426,74",2026-08-09 16:14:08
4,f3c3fa72-8783-4b35-92bb-6cd4674c745a,Srta. Clarice Moraes,lda-cunha@example.org,07/07/2026,Martins do Amparo,sao paulo,"R$ 96,35",2026-07-21 01:34:11



--- Últimas 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
210,167b6dba-1bae-4ce9-a173-d75867d903d5,Raquel Machado,ocorreia@example.com,03-11-2026,Macedo,SC,NaN,2026-08-05 11:49:54
211,d740ef8b-b355-4f04-b3ab-9fa04015cb50,João Vitor da Cunha,marcos-viniciusmoraes@example.org,2025-11-13,Aragão,RJ,"R$ 87,72",2026-08-18 04:46:48
212,18e87cd6-4519-43ed-9748-4db8ad7e72be,Sr. João Guilherme Brito,ccirino@example.net,02-11-2025,Pastor,BA,NaN,2026-07-25 01:20:33
213,7b9fa8a1-ad1c-4922-8edb-ae5c8795eb45,Luiz Gustavo Pereira,ibarros@example.net,14/02/2025,Jesus de Câmara,BA,Não informado,2026-08-03 04:42:21
214,24bde8c8-52f0-4f22-a082-34f735108e0e,Mariah Moura,da-cruzrebeca@example.com,29-Mar-2025,Santos,MG,"R$ 341,86",2026-08-14 00:56:13



--- Amostra aleatória de 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
91,af757bcc-8928-477c-a8d0-841c4a6a9934,Diogo Rocha,maria-alice46@example.net,29/08/2025,Aparecida do Oeste,RJ,"R$ 584,29",2026-08-12 02:01:39
60,9192bcd4-64d1-44a1-998c-8783083b55c9,Thales Freitas,antonioviana@example.com,08/07/2026,Monteiro do Norte,SP,"R$ 427,69",2026-07-29 15:18:26
155,f90b07b0-a209-4d5e-ae39-3e85d5ef4961,Anthony Sousa,egomes@example.net,09-Feb-2026,Correia de Minas,MG,"R$ 69,29",2026-08-14 18:21:31
118,ba2a9dee-bf29-4197-bade-a7184dba6e4c,Maria da Paz,pintoisaque@example.net,09/06/2025,Porto,BA,"R$ 111,48",2026-07-24 03:19:35
73,9f1dd824-1393-4327-834a-f2c0b3b6acca,Ryan Peixoto,valentina58@example.com,30/11/2024,Nogueira de Mendes,SC,NaN,2026-08-02 05:53:36


#### Usando `.info()` para um Resumo Técnico

O método `.info()` é um dos nossos melhores amigos. Ele nos dá um resumo conciso do DataFrame, incluindo:
- O número total de entradas (linhas).
- O número de colunas.
- O nome e a contagem de valores **não nulos** para cada coluna.
- O **tipo de dado (`Dtype`)** de cada coluna.
- O uso de memória.

**O que procurar aqui?**
1.  **Contagem de Não Nulos:** Se o valor for menor que o total de entradas, significa que a coluna tem dados faltantes.
2.  **Dtype:** O tipo de dado está correto? Uma coluna de valor de compra deveria ser numérica (`float64` ou `int64`), não `object` (que geralmente significa string). Uma coluna de data deveria ser `datetime64[ns]`, não `object`.

In [20]:
# Obtendo um resumo técnico do DataFrame
df_usuarios.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   user_id              215 non-null    str  
 1   nome                 215 non-null    str  
 2   email                202 non-null    str  
 3   data_cadastro        215 non-null    str  
 4   cidade               215 non-null    str  
 5   estado               215 non-null    str  
 6   valor_ultima_compra  188 non-null    str  
 7   data_ultimo_login    215 non-null    str  
dtypes: str(8)
memory usage: 39.9 KB


#### Usando `.describe(include='all')` para Estatísticas Descritivas

Enquanto `.info()` nos dá a estrutura, `.describe()` nos dá um resumo estatístico. Usando `include='all'`, forçamos o Pandas a nos mostrar estatísticas tanto para colunas numéricas quanto para as de texto (categóricas).

- **Para colunas numéricas:** `count`, `mean` (média), `std` (desvio padrão), `min`, `max`, e os quartis (`25%`, `50%`, `75%`).
- **Para colunas de objeto/categóricas:** `count`, `unique` (número de valores únicos), `top` (valor mais frequente), e `freq` (frequência do valor mais frequente).

In [21]:
# Obtendo um resumo estatístico de todas as colunas
df_usuarios.describe(include='all')

,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
count,215,215,202,215,215,215,188,215
unique,200,200,186,193,152,10,168,200
top,47d04004-4fc2-49e1-9fad-cea7dd7847d7,Evelyn Fogaça,rfonseca@example.com,24/11/2024,Santos,SC,Não informado,2026-07-31 06:16:13
freq,2,2,2,2,5,40,9,2


### ✏️ Exercício 1: Análise Pós-Inspeção

Com base nas saídas dos comandos `.info()` e `.describe(include='all')`, responda na célula abaixo às seguintes perguntas:

1.  Quais colunas têm valores faltantes? A contagem de não nulos em `.info()` te deu essa resposta.
2.  A coluna `valor_ultima_compra` é do tipo correto para realizarmos cálculos (como a média)?
3.  As colunas `data_cadastro` e `data_ultimo_login` estão em um formato de data que o pandas entende nativamente?
4.  Olhando para a estatística `unique` da coluna `estado` no `.describe()`, o número parece alto ou baixo demais? O que isso pode indicar?

Responda aqui.

1. As colunas `email` e `valor_ultima_compra` possuem valores faltantes.
2. Não. `valor_ultima_compra` está como `object`, pois seus valores estão armazenados como texto.
3. Não. `data_cadastro` e `data_ultimo_login` ainda estão como texto (`object`).
4. O número pode parecer alto porque existem variações de escrita para o mesmo estado, como `SP`, `São Paulo` e `sao paulo`. Isso indica falta de padronização nos dados categóricos.

### 2.3 Tratando Dados Faltantes (Valores Nulos)

Nossa investigação revelou que as colunas `email` e `valor_ultima_compra` têm valores faltantes. Vamos lidar com eles.

In [22]:
# Contando o número de valores nulos em cada coluna
df_usuarios.isnull().sum()

user_id                 0
nome                    0
email                  13
data_cadastro           0
cidade                  0
estado                  0
valor_ultima_compra    27
data_ultimo_login       0
dtype: int64

#### Estratégias para Lidar com Dados Faltantes

Existem duas estratégias principais para lidar com dados faltantes:

1.  **Remoção:** Excluir as linhas (ou colunas) que contêm valores faltantes. É uma abordagem rápida e simples, mas tem um custo: a perda de dados. Se uma linha tem um valor importante faltando, mas as outras informações são valiosas, removê-la pode não ser o ideal.
2.  **Imputação (Preenchimento):** Preencher os valores faltantes com um valor estimado. Pode ser um valor fixo (como 0), a média, a mediana ou a moda da coluna. Esta abordagem preserva o resto dos dados da linha, mas introduz um valor que não é original.

A escolha da estratégia depende do contexto do negócio e da natureza da coluna.

#### Estratégia 1: Remover Linhas com `dropna()`

**Cenário:** Um usuário sem e-mail é de pouca utilidade para nosso e-commerce. Não podemos contatá-lo para marketing, recuperação de senha ou confirmação de pedidos. Portanto, a decisão de negócio aqui é **remover** os cadastros que não possuem um e-mail.

Usaremos `df.dropna(subset=['nome_da_coluna'])` para remover apenas as linhas onde o valor na coluna especificada é nulo.

In [23]:
# Verificando o número de linhas antes da remoção
print(f"Número de linhas antes de remover nulos em 'email': {len(df_usuarios)}")

# Contando os nulos em 'email' para confirmar
print(f"Número de valores nulos em 'email': {df_usuarios['email'].isnull().sum()}\n")

# Removendo as linhas onde a coluna 'email' é nula
df_usuarios = df_usuarios.dropna(subset=['email']).copy()

# Verificando o número de linhas depois da remoção
print(f"Número de linhas após remover nulos em 'email': {len(df_usuarios)}")

# Confirmando que não há mais nulos em 'email'
print(f"Número de valores nulos em 'email' agora: {df_usuarios['email'].isnull().sum()}")

Número de linhas antes de remover nulos em 'email': 215
Número de valores nulos em 'email': 13

Número de linhas após remover nulos em 'email': 202
Número de valores nulos em 'email' agora: 0


#### Estratégia 2: Imputar Valores com `fillna()`

**Cenário:** A coluna `valor_ultima_compra` também tem valores nulos. No entanto, remover essas linhas significaria perder informações de usuários que, embora não tenham um valor de compra registrado, ainda são clientes cadastrados. Uma abordagem melhor é a **imputação**.

**Média vs. Mediana:** Qual valor usar para preencher?
- **Média (`mean`):** A soma de todos os valores dividida pelo número de valores. É muito sensível a *outliers* (valores extremamente altos ou baixos). Uma única compra de valor muito alto poderia inflar a média e distorcer a realidade.
- **Mediana (`median`):** O valor do meio quando todos os dados são ordenados. É robusta a outliers e, por isso, é geralmente a escolha mais segura para dados financeiros ou com distribuição assimétrica, como valores de compra.

Vamos tentar calcular a mediana e preencher os valores nulos. Mas... há um problema!

In [24]:
try:
    mediana_compra = df_usuarios['valor_ultima_compra'].median()
    print(f"Mediana calculada: {mediana_compra}")
except TypeError as e:
    print(f"Ocorreu um erro: {e}")
    print("\nNão podemos calcular a mediana de uma coluna que não é numérica! Isso nos leva ao próximo passo.")

Ocorreu um erro: Cannot perform reduction 'median' with string dtype

Não podemos calcular a mediana de uma coluna que não é numérica! Isso nos leva ao próximo passo.


### 2.4 Corrigindo Tipos de Dados

O erro acima aconteceu porque, como vimos no `.info()`, a coluna `valor_ultima_compra` é do tipo `object` (string), e não um número. Precisamos convertê-la.

#### Convertendo `valor_ultima_compra` para Numérico

Para converter, precisamos primeiro limpar a string, removendo o `R$ ` e trocando a vírgula decimal por um ponto. Depois, usamos `pd.to_numeric`.

O parâmetro `errors='coerce'` é muito útil: ele transformará qualquer valor que não possa ser convertido em um número (como a string 'Não informado') em `NaN`. Isso é ótimo, pois podemos tratar todos os problemas de uma vez só.

In [25]:
# Passo 1: Limpar a string
df_usuarios['valor_ultima_compra'] = (
    df_usuarios['valor_ultima_compra']
    .astype('string')
    .str.replace('R$ ', '', regex=False)
    .str.replace(',', '.', regex=False)
    .replace({'Não informado': np.nan})
    .str.strip()
 )

# Passo 2: Converter para tipo numérico, tratando erros
df_usuarios['valor_ultima_compra'] = pd.to_numeric(
    df_usuarios['valor_ultima_compra'], errors='coerce'
 )

# Vamos verificar o tipo de dado da coluna agora
print("Tipo de dado de 'valor_ultima_compra' após conversão:")
print(df_usuarios.dtypes['valor_ultima_compra'])

# E ver como ficaram os 10 primeiros valores
print("\nValores após conversão (note os novos NaNs onde antes era 'Não informado'):")
display(df_usuarios[['nome', 'valor_ultima_compra']].head(10))

Tipo de dado de 'valor_ultima_compra' após conversão:
Float64

Valores após conversão (note os novos NaNs onde antes era 'Não informado'):


,nome,valor_ultima_compra
0,Marcos Vinicius Silveira,<NA>
1,Srta. Isabel Vieira,647.44
2,Valentim da Conceição,618.83
3,Sr. Théo Cavalcanti,426.74
4,Srta. Clarice Moraes,96.35
5,Gabriela Vargas,872.34
7,Bernardo Porto,486.98
8,Catarina Aparecida,880.02
9,Kevin Cavalcante,904.84
10,Luísa Farias,750.09


#### Agora sim: Imputando a Mediana

Com a coluna `valor_ultima_compra` agora no formato `float64`, podemos finalmente calcular a mediana e usar `fillna()` para preencher todos os valores `NaN` (os que já existiam e os que foram criados pelo `errors='coerce'`).

In [26]:
# Verificando nulos ANTES da imputação
print(f"Nulos em 'valor_ultima_compra' ANTES da imputação: {df_usuarios['valor_ultima_compra'].isnull().sum()}")

# 1. Calcular a mediana (agora vai funcionar!)
mediana_compra = df_usuarios['valor_ultima_compra'].median()

print(f"A mediana calculada é: R$ {mediana_compra:.2f}")

# 2. Preencher os valores nulos com a mediana
df_usuarios['valor_ultima_compra'] = df_usuarios['valor_ultima_compra'].fillna(mediana_compra)

# Verificando nulos depois da imputação
print(f"Nulos em 'valor_ultima_compra' APÓS a imputação: {df_usuarios['valor_ultima_compra'].isnull().sum()}")

Nulos em 'valor_ultima_compra' ANTES da imputação: 34
A mediana calculada é: R$ 511.49
Nulos em 'valor_ultima_compra' APÓS a imputação: 0


#### Convertendo Colunas de Data

As colunas `data_cadastro` e `data_ultimo_login` também são do tipo `object`. Precisamos convertê-las para o tipo `datetime` para que possamos realizar operações com datas, como calcular a quanto tempo um usuário se cadastrou.

A função `pd.to_datetime` é extremamente poderosa. Para a `data_cadastro`, que tem vários formatos, podemos usar o argumento `format='mixed'` para que o Pandas tente adivinhar o formato correto para cada linha.

Novamente, usaremos `errors='coerce'` para converter qualquer data que não possa ser entendida em `NaT` (Not a Time), o equivalente a `NaN` para datas.

In [27]:
# Convertendo a coluna 'data_cadastro' para datetime
# 'format="mixed"' permite que o pandas tente adivinhar múltiplos formatos
df_usuarios['data_cadastro'] = pd.to_datetime(
    df_usuarios['data_cadastro'], format='mixed', errors='coerce'
 )

# A coluna 'data_ultimo_login' tem um formato mais consistente, mas ainda é object
df_usuarios['data_ultimo_login'] = pd.to_datetime(
    df_usuarios['data_ultimo_login'], errors='coerce'
 )

# Vamos verificar os tipos de dados novamente com.info()
print("--- Verificação dos Dtypes após conversão de datas ---")
df_usuarios.info()

--- Verificação dos Dtypes após conversão de datas ---
<class 'pandas.DataFrame'>
Index: 202 entries, 0 to 214
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              202 non-null    str           
 1   nome                 202 non-null    str           
 2   email                202 non-null    str           
 3   data_cadastro        202 non-null    datetime64[us]
 4   cidade               202 non-null    str           
 5   estado               202 non-null    str           
 6   valor_ultima_compra  202 non-null    Float64       
 7   data_ultimo_login    202 non-null    datetime64[us]
dtypes: Float64(1), datetime64[us](2), str(5)
memory usage: 32.2 KB


Sucesso! As colunas `valor_ultima_compra`, `data_cadastro` e `data_ultimo_login` agora têm os tipos de dados corretos (`float64` e `datetime64[ns]`), como esperado. Agora podemos fazer operações matemáticas e de data, como calcular o tempo desde o último login ou a média de compras.

### ✏️ Exercício 2: Cálculos Pós-Conversão

Agora que os tipos de dados estão corretos, realize as seguintes tarefas em células de código separadas:

1.  Calcule o valor **médio** da coluna `valor_ultima_compra` e imprima o resultado formatado.
2.  Encontre a data do **último login mais recente** em todo o dataset .
3.  Calcule há quantos dias foi o último login do **primeiro usuário** do DataFrame.

In [28]:
# 1. Calcule o valor médio da coluna 'valor_ultima_compra'
media_compra = df_usuarios['valor_ultima_compra'].mean()
print(f"Valor médio da última compra: R$ {media_compra:.2f}")

Valor médio da última compra: R$ 509.55


In [29]:
# 2. Encontre a data do último login mais recente
ultimo_login = df_usuarios['data_ultimo_login'].max()
print(f"Último login mais recente: {ultimo_login.strftime('%d/%m/%Y %H:%M:%S')}")

Último login mais recente: 19/08/2026 10:58:01


In [30]:
# 3. Calcule há quantos dias foi o último login do primeiro usuário
primeiro_login = df_usuarios.iloc[0]['data_ultimo_login']
dias_desde_primeiro_login = (pd.Timestamp.now() - primeiro_login).days
print(f"O último login do primeiro usuário ocorreu há {dias_desde_primeiro_login} dias.")

O último login do primeiro usuário ocorreu há 18 dias.


---

### 2.5 Removendo Duplicatas

Dados duplicados podem distorcer análises, como a contagem de usuários únicos. Vamos verificar se existem e removê-los.

- `.duplicated().sum()`: Conta quantas linhas são duplicatas exatas de outras que já apareceram.
- `.drop_duplicates()`: Retorna um DataFrame com as duplicatas removidas.

In [31]:
# Verificando o número de linhas duplicadas
num_duplicatas = df_usuarios.duplicated().sum()
print(f"Número de linhas duplicadas encontradas: {num_duplicatas}")

# Removendo as duplicatas
print(f"Linhas antes de remover duplicatas: {len(df_usuarios)}")
df_usuarios = df_usuarios.drop_duplicates().reset_index(drop=True)

print(f"Linhas após remover duplicatas: {len(df_usuarios)}")

Número de linhas duplicadas encontradas: 15
Linhas antes de remover duplicatas: 202
Linhas após remover duplicatas: 187


### 2.6 Padronizando Dados Categóricos

O último passo da nossa limpeza é garantir que dados de texto (categóricos) sejam consistentes. Na nossa inspeção, suspeitamos da coluna `estado`.

Vamos usar `.unique()` para ver todos os valores distintos que a coluna possui.

In [32]:
# Verificando os valores únicos na coluna 'estado'
print("Valores únicos em 'estado' ANTES da padronização:")
print(df_usuarios['estado'].unique())

# Criando o dicionário de mapeamento para corrigir as inconsistências
mapa_estados = {
    'São Paulo': 'SP',
    'sao paulo': 'SP',
    'Rio de Janeiro': 'RJ',
    'rio de janeiro': 'RJ'
    # Não precisamos mapear 'SP' -> 'SP' ou 'RJ' -> 'RJ', o replace ignora chaves que não encontra
}

# Aplicando a substituição
df_usuarios['estado'] = df_usuarios['estado'].replace(mapa_estados)

# Verificando os valores únicos novamente para confirmar a limpeza
print("\nValores únicos em 'estado' APÓS a padronização:")
print(df_usuarios['estado'].unique())

Valores únicos em 'estado' ANTES da padronização:
<ArrowStringArray>
[     'São Paulo',             'MG',             'PR',             'BA',
      'sao paulo',             'RJ',             'SC', 'Rio de Janeiro',
             'SP', 'rio de janeiro']
Length: 10, dtype: str

Valores únicos em 'estado' APÓS a padronização:
<ArrowStringArray>
['SP', 'MG', 'PR', 'BA', 'RJ', 'SC']
Length: 6, dtype: str


Excelente! Agora nossa coluna `estado` está limpa e padronizada, pronta para ser usada em análises de agrupamento (`groupby`).

### 2.7 Salvando o Trabalho

Missão cumprida! Passamos por todas as etapas do nosso fluxo de trabalho de limpeza. O passo final é salvar nosso DataFrame limpo em um novo arquivo CSV. Este arquivo será o ponto de partida para futuras análises.

Vamos dar uma última olhada no nosso trabalho com `.info()` para confirmar que tudo está em ordem: sem nulos e com os tipos de dados corretos.

In [33]:
# Verificação final
df_usuarios.info()

<class 'pandas.DataFrame'>
RangeIndex: 187 entries, 0 to 186
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              187 non-null    str           
 1   nome                 187 non-null    str           
 2   email                187 non-null    str           
 3   data_cadastro        187 non-null    datetime64[us]
 4   cidade               187 non-null    str           
 5   estado               187 non-null    str           
 6   valor_ultima_compra  187 non-null    Float64       
 7   data_ultimo_login    187 non-null    datetime64[us]
dtypes: Float64(1), datetime64[us](2), str(5)
memory usage: 28.0 KB


In [34]:
# Salvando o DataFrame limpo em um novo arquivo CSV
df_usuarios.to_csv('usuarios_limpo.csv', index=False)

print("Arquivo 'usuarios_limpo.csv' salvo com sucesso!")

Arquivo 'usuarios_limpo.csv' salvo com sucesso!


---
## Conclusão

Parabéns! Você completou um ciclo completo de limpeza de dados. Você pegou um dataset caótico e, aplicando um método sistemático, o transformou em uma fonte de dados organizada e confiável.

**O que nós fizemos:**
- **Inspecionamos** os dados para encontrar problemas.
- **Tratamos valores faltantes** usando duas estratégias diferentes (remoção e imputação).
- **Corrigimos tipos de dados** incorretos, permitindo cálculos e operações.
- **Removemos dados duplicados** para garantir a unicidade dos registros.
- **Padronizamos dados categóricos** para permitir agrupamentos e análises consistentes.

Agora, com o arquivo `usuarios_limpo.csv` em mãos, você está pronto para a próxima aula, onde vamos explorar e visualizar esses dados para descobrir padrões e insights de negócio.